# 02 - Detection data pipeline

This notebook inspects the Dataset, letterbox transform, and DataLoaders used by the object detection pipeline. The implementation stays in `src/`; the notebook only visualizes its outputs.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import torch

project_root = Path.cwd()
while not (project_root / 'data').exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

from turtle_detection.data_pipeline import create_data_loaders

print(f'Project root: {project_root}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')

In [ ]:
train_loader, validation_loader = create_data_loaders(
    project_root,
    batch_size=4,
    num_workers=0,
)
print(f'Train images: {len(train_loader.dataset)}')
print(f'Validation images: {len(validation_loader.dataset)}')
print(f'Train batches: {len(train_loader)}')
print(f'Validation batches: {len(validation_loader)}')

In [ ]:
def show_detection_batch(loader, title, max_images=4):
    images, targets = next(iter(loader))
    count = min(max_images, len(images))
    figure, axes = plt.subplots(1, count, figsize=(4 * count, 4))
    if count == 1:
        axes = [axes]
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    for axis, image, target in zip(axes, images[:count], targets[:count]):
        display_image = (image.cpu() * std + mean).clamp(0, 1)
        display_image = display_image.permute(1, 2, 0).numpy()
        axis.imshow(display_image)
        box = target['boxes'][0].numpy()
        x1, y1, x2, y2 = box
        axis.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='lime', linewidth=2))
        axis.set_title(target['image_id'])
        axis.axis('off')
    figure.suptitle(title)
    figure.tight_layout()
    plt.show()

In [ ]:
show_detection_batch(train_loader, 'Train samples with random horizontal flip')
show_detection_batch(validation_loader, 'Validation samples without augmentation')

## What to verify

- The output tensor is `(3, 384, 512)` and uses ImageNet normalization.
- Bounding boxes remain aligned after letterbox padding and train-time flipping.
- Validation is deterministic: no horizontal flip is applied.
- Detection targets are lists, so a future model can handle variable numbers of boxes per image.
- `num_workers=0` is intentional for notebooks and native Windows; it avoids multiprocessing spawn issues while the pipeline is being validated.